# 2526_Pràctica ANN, CNN i transfer learning

# Agafarem el dataset cifar-10 de tensorflow-datasets:

In [3]:
!pip install tensorflow-datasets
!pip install importlib-resources

In [8]:
  

# Normalizar
X_train = X_train / 255.0
X_test = X_test / 255.0

# Aplanar para ANN
X_train = X_train.reshape(-1, 32*32*3)
X_test = X_test.reshape(-1, 32*32*3)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 451s 3us/step


c:\Users\EIABD\miniconda3\envs\tensorflow\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


In [11]:
X_test

array([[0.61960784, 0.43921569, 0.19215686, ..., 0.08235294, 0.2627451 ,
        0.43137255],
       [0.92156863, 0.92156863, 0.92156863, ..., 0.72941176, 0.78431373,
        0.78039216],
       [0.61960784, 0.74509804, 0.87058824, ..., 0.02745098, 0.03137255,
        0.02745098],
       ...,
       [0.07843137, 0.05882353, 0.04705882, ..., 0.09803922, 0.07843137,
        0.18431373],
       [0.09803922, 0.15686275, 0.04705882, ..., 0.36078431, 0.47058824,
        0.31372549],
       [0.28627451, 0.30588235, 0.29411765, ..., 0.10588235, 0.10196078,
        0.10196078]], shape=(10000, 3072))

# ANN. Construeix una xarxa neuronal feed forward 

Provis diverses geometries per veure quina s'ajusta millor.
Empris callbacks i capes vistes a classe, incloent Dropout

In [16]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3
    )
]

In [14]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(512, activation='relu', input_shape=(3072,)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(10, activation='softmax')
])
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                      │ (None, 512)                 │       1,573,376 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 10)                  │           5,130 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,578,506 (6.02 MB)

 Trainable params: 1,578,506 (6.02 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [17]:
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=64,
    callbacks=callbacks
)

Epoch 1/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.2727 - loss: 2.0187 - val_accuracy: 0.3493 - val_loss: 1.8458 - learning_rate: 0.0010
Epoch 2/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.3189 - loss: 1.8736 - val_accuracy: 0.3664 - val_loss: 1.7974 - learning_rate: 0.0010
Epoch 3/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.3346 - loss: 1.8400 - val_accuracy: 0.3715 - val_loss: 1.7672 - learning_rate: 0.0010
Epoch 4/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.3393 - loss: 1.8223 - val_accuracy: 0.3765 - val_loss: 1.7441 - learning_rate: 0.0010
Epoch 5/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.3528 - loss: 1.7964 - val_accuracy: 0.3783 - val_loss: 1.7320 - learning_rate: 0.0010
Epoch 6/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.3553 - loss: 1.7818 - val_accuracy: 0.3720 - val_loss: 1.7310 - learning_rate: 0.0010
Epoch 7/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.3535 - loss: 1.

In [18]:
test = model.evaluate(X_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4471 - loss: 1.5612


In [24]:
model2 = tf.keras.Sequential([
    tf.keras.layers.Dense(1024, activation='relu', input_shape=(3072,)),
    tf.keras.layers.Dropout(0.4),

    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(10, activation='softmax')
])

model2.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model2.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=64,
    callbacks=callbacks
)

Epoch 1/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 15s 23ms/step - accuracy: 0.2267 - loss: 2.0826 - val_accuracy: 0.2860 - val_loss: 1.9230 - learning_rate: 0.0010
Epoch 2/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - accuracy: 0.2676 - loss: 1.9645 - val_accuracy: 0.3082 - val_loss: 1.9353 - learning_rate: 0.0010
Epoch 3/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - accuracy: 0.2804 - loss: 1.9236 - val_accuracy: 0.3232 - val_loss: 1.8787 - learning_rate: 0.0010
Epoch 4/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 15s 24ms/step - accuracy: 0.3146 - loss: 1.8567 - val_accuracy: 0.3599 - val_loss: 1.8303 - learning_rate: 5.0000e-04
Epoch 5/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 15s 24ms/step - accuracy: 0.3197 - loss: 1.8386 - val_accuracy: 0.3425 - val_loss: 1.8601 - learning_rate: 5.0000e-04


In [25]:
test2 = model2.evaluate(X_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.2875 - loss: 1.9096


In [21]:
model3 = tf.keras.Sequential([
    tf.keras.layers.Dense(2048, activation='relu', input_shape=(3072,)),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(1024, activation='relu'),
    tf.keras.layers.Dropout(0.4),

    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(10, activation='softmax')
])

model3.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history3 = model3.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=64,
    callbacks=callbacks
)

test3 = model3.evaluate(X_test, y_test)

Epoch 1/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 36s 57ms/step - accuracy: 0.2040 - loss: 2.1438 - val_accuracy: 0.2783 - val_loss: 2.0010 - learning_rate: 0.0010
Epoch 2/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 35s 56ms/step - accuracy: 0.2431 - loss: 2.0127 - val_accuracy: 0.3024 - val_loss: 1.9383 - learning_rate: 0.0010
Epoch 3/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 36s 57ms/step - accuracy: 0.2528 - loss: 1.9806 - val_accuracy: 0.2997 - val_loss: 1.9484 - learning_rate: 0.0010
Epoch 4/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 35s 56ms/step - accuracy: 0.2837 - loss: 1.9225 - val_accuracy: 0.3175 - val_loss: 1.9268 - learning_rate: 5.0000e-04
Epoch 5/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 35s 56ms/step - accuracy: 0.2940 - loss: 1.8988 - val_accuracy: 0.3213 - val_loss: 1.9381 - learning_rate: 5.0000e-04
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2822 - loss: 1.9928


In [27]:
print(test, test2, test3)

[1.5612444877624512, 0.4471000134944916] [1.9096001386642456, 0.2874999940395355] [1.992846131324768, 0.28220000863075256]


# CNN
Provis també diverses geometries, especialment de capes de convolució i pooling.

Provis paràmetres distints a les capes de convolució (mides de kernels, mida de passa, relleno...)

In [28]:
# Volvemos a cargar los datos ya que para ANN tuvimos que hacer reshape para transformarlo en vectores
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()

X_train = X_train / 255.0
X_test = X_test / 255.0


In [31]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3)
]

In [29]:
model1 = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

c:\Users\EIABD\miniconda3\envs\tensorflow\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [33]:
model1.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [34]:
history = model1.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=64,
    callbacks=callbacks
)

Epoch 1/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.4284 - loss: 1.5854 - val_accuracy: 0.5428 - val_loss: 1.3019 - learning_rate: 0.0010
Epoch 2/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.5795 - loss: 1.1945 - val_accuracy: 0.6213 - val_loss: 1.1068 - learning_rate: 0.0010
Epoch 3/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.6323 - loss: 1.0513 - val_accuracy: 0.6409 - val_loss: 1.0349 - learning_rate: 0.0010
Epoch 4/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.6679 - loss: 0.9545 - val_accuracy: 0.6457 - val_loss: 1.0143 - learning_rate: 0.0010
Epoch 5/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.6867 - loss: 0.8970 - val_accuracy: 0.6693 - val_loss: 0.9698 - learning_rate: 0.0010
Epoch 6/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.7087 - loss: 0.8360 - val_accuracy: 0.6734 - val_loss: 0.9746 - learning_rate: 0.0010
Epoch 7/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.7262 - loss: 0.

In [35]:
test1 = model1.evaluate(X_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7116 - loss: 0.8968


In [36]:
model2 = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10, activation='softmax')
])
model2.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
history = model2.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=64,
    callbacks=callbacks
)

Epoch 1/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 31ms/step - accuracy: 0.3915 - loss: 1.6651 - val_accuracy: 0.5397 - val_loss: 1.2821 - learning_rate: 0.0010
Epoch 2/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 19s 30ms/step - accuracy: 0.5547 - loss: 1.2493 - val_accuracy: 0.6466 - val_loss: 1.0106 - learning_rate: 0.0010
Epoch 3/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 19s 31ms/step - accuracy: 0.6292 - loss: 1.0582 - val_accuracy: 0.6722 - val_loss: 0.9172 - learning_rate: 0.0010
Epoch 4/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 19s 31ms/step - accuracy: 0.6929 - loss: 0.8818 - val_accuracy: 0.7087 - val_loss: 0.8257 - learning_rate: 1.0000e-04
Epoch 5/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.7035 - loss: 0.8484 - val_accuracy: 0.7098 - val_loss: 0.8142 - learning_rate: 1.0000e-04
Epoch 6/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 19s 31ms/step - accuracy: 0.7122 - loss: 0.8282 - val_accuracy: 0.7194 - val_loss: 0.7953 - learning_rate: 1.0000e-04
Epoch 7/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 19s 31ms/step - accuracy

In [37]:
test2 = model2.evaluate(X_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7526 - loss: 0.7451


In [38]:
model3 = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (5,5), strides=(1,1), padding='valid',
                           activation='relu', input_shape=(32,32,3)),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(64, (3,3), strides=(2,2), padding='same',
                           activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])
model3.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
history = model3.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=64,
    callbacks=callbacks
)

Epoch 1/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.4047 - loss: 1.6404 - val_accuracy: 0.5020 - val_loss: 1.4018 - learning_rate: 0.0010
Epoch 2/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.5221 - loss: 1.3313 - val_accuracy: 0.5587 - val_loss: 1.2650 - learning_rate: 0.0010
Epoch 3/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.5764 - loss: 1.2000 - val_accuracy: 0.5752 - val_loss: 1.1972 - learning_rate: 0.0010
Epoch 4/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.6298 - loss: 1.0552 - val_accuracy: 0.6110 - val_loss: 1.1294 - learning_rate: 1.0000e-04
Epoch 5/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.6380 - loss: 1.0333 - val_accuracy: 0.6151 - val_loss: 1.1067 - learning_rate: 1.0000e-04


In [39]:
test3 = model3.evaluate(X_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5016 - loss: 1.3925


In [40]:
print(test, test2, test3)

[1.5612444877624512, 0.4471000134944916] [0.745083212852478, 0.7526000142097473] [1.3925106525421143, 0.5016000270843506]


# Transfer-learning i fine-tunning
Agafa la millor xarxa neuronal dels apartats anteriors. 
    
Transfereix el coneixement a una xarxa neuronal nova amb una geometria lleugerament diferents (per exemple una capa profunda més).

Següeix les passes vistes a classe per tal de transferir el coneixement sense destruir l'anterior i acabar d'ajustar el model nou a l'antic amb el mètode apropiat


In [41]:
base_model = model2

# Congelar el modelo base
for layer in base_model.layers:
    layer.trainable = False

In [42]:
# Capa extra
new_model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(10, activation='softmax')
])

In [43]:
new_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

new_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.7662 - loss: 1.0057 - val_accuracy: 0.7613 - val_loss: 0.7924
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.8196 - loss: 0.5921 - val_accuracy: 0.7622 - val_loss: 0.7998
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.8189 - loss: 0.5842 - val_accuracy: 0.7614 - val_loss: 0.7956
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.8187 - loss: 0.5824 - val_accuracy: 0.7627 - val_loss: 0.7945
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.8188 - loss: 0.5813 - val_accuracy: 0.7622 - val_loss: 0.7941
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.8204 - loss: 0.5758 - val_accuracy: 0.7617 - val_loss: 0.7917
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.8181 - loss: 0.5792 - val_accuracy: 0.7618 - val_loss: 0.7910
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.8209 - loss: 0.5718 - val_accu

In [44]:
# fine-tuning
# desbloqueamos solo las ultimas capas del modelo original
for layer in base_model.layers[-4:]:  # ultimas capas
    layer.trainable = True

In [ ]:
# bajar learning rate
new_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [45]:
new_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8209 - loss: 0.5636 - val_accuracy: 0.7624 - val_loss: 0.7857
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8189 - loss: 0.5671 - val_accuracy: 0.7619 - val_loss: 0.7828
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8217 - loss: 0.5544 - val_accuracy: 0.7615 - val_loss: 0.7847
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8191 - loss: 0.5611 - val_accuracy: 0.7611 - val_loss: 0.7791
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8201 - loss: 0.5571 - val_accuracy: 0.7619 - val_loss: 0.7761
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.8220 - loss: 0.5541 - val_accuracy: 0.7615 - val_loss: 0.7781
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.8204 - loss: 0.5499 - val_accuracy: 0.7617 - val_loss: 0.7794
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8205 - loss: 0.5520 - val_accu

In [47]:
test = new_model.evaluate(X_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7528 - loss: 0.8175


Se ha reutilizado el modelo previamente entrenado como base, congelando sus capas para preservar el conocimiento adquirido. Se ha añadido una nueva capa densa para incrementar la capacidad del modelo y posteriormente se ha realizado fine-tuning desbloqueando parcialmente las capas finales con un learning rate reducido